In [1]:
print ("yoo anaconda working?")

yoo anaconda working?


In [1]:
print ("yoo still")
2+2

yoo still


4

In [3]:
import pandas as pd
import numpy as np
print ("okay it works now")

okay it works now


In [8]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")

db = client['houseprices']
collection = db['predictions']

# Fake prediction doc (what API will save)
doc = {
    "neighborhood": "Nyarutarama",
    "plot_size_sqm": 600,
    "bedrooms": 5,
    "bathrooms": 4,
    "predicted_price_millions_rwf": 420.75,
    "price_range": "378.7 - 462.8 million RWF",
    "inserted_at": "2026-03-05"
}

inserted = collection.insert_one(doc)

print("Inserted document ID:", inserted.inserted_id)
print("Total docs now:", collection.count_documents({}))

Inserted document ID: 69a9357d58a59a0cef006082
Total docs now: 1


In [1]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client['houseprices']
collection = db['predictions']
print("Connected! Databases:", client.list_database_names())

Connected! Databases: ['admin', 'config', 'houseprices', 'local']


In [24]:
from pymongo import MongoClient
import datetime 
client = MongoClient("mongodb://localhost:27017/")
print("connected,list of databases:",
      client.list_database_names())
db = client['houseprices']
collections = db['predictions']
test_doc = {
    "test_message":"working 808",
    "neighborhood_example": "kiyovu",
    "fake_price": 350.0,
    "inserted_at": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}
result = collection.insert_one(test_doc)
print("inserted document ID:",result.inserted_id)
print("total documents in predictions now",
collections.count_documents({}))
print("quick find one:", collection.find_one())

connected,list of databases: ['admin', 'config', 'houseprices', 'local']
inserted document ID: 69a941b76c176d708ac4981b
total documents in predictions now 3
quick find one: {'_id': ObjectId('69a9357d58a59a0cef006082'), 'neighborhood': 'Nyarutarama', 'plot_size_sqm': 600, 'bedrooms': 5, 'bathrooms': 4, 'predicted_price_millions_rwf': 420.75, 'price_range': '378.7 - 462.8 million RWF', 'inserted_at': '2026-03-05'}


In [26]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

np.random.seed(42)
n = 200
neighborhoods = ['Kiyovu', 'Nyarutarama', 'Kimihurura', 'Kimironko', 'Nyamirambo', 'Kacyiru', 'Remera', 'Gisozi']
data = {
    'neighborhood': np.random.choice(neighborhoods, n),
    'plot_size_sqm': np.random.uniform(200, 1200, n).round(0),
    'bedrooms': np.random.randint(2, 7, n),
    'bathrooms': np.random.randint(1, 5, n),
}
df = pd.DataFrame(data)

premium = {'Kiyovu': 1.6, 'Nyarutarama': 1.5, 'Kimihurura': 1.4, 'Kacyiru': 1.3,
           'Remera': 1.1, 'Gisozi': 1.0, 'Kimironko': 0.9, 'Nyamirambo': 0.8}
df['price_millions_rwf'] = (
    (df['plot_size_sqm'] * 0.15) +
    (df['bedrooms'] * 20) +
    (df['bathrooms'] * 15) + 50
) * df['neighborhood'].map(premium) * np.random.uniform(0.85, 1.15, n)
df['price_millions_rwf'] = df['price_millions_rwf'].round(1)

print(df.head())

  neighborhood  plot_size_sqm  bedrooms  bathrooms  price_millions_rwf
0       Remera          231.0         6          2               259.7
1    Kimironko          836.0         5          4               326.5
2   Nyamirambo          514.0         4          3               202.9
3       Remera          709.0         5          3               366.5
4   Kimihurura         1108.0         6          1               499.3


In [1]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

# Features and target
X = df.drop('price_millions_rwf', axis=1)
y = df['price_millions_rwf']

# Preprocessor
cat_features = ['neighborhood']
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ],
    remainder='passthrough'  # keep the numbers as-is
)

# Full pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Split and train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)

# Predict & score
preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print(f"MAE: {mae:.2f} million RWF (average error)")
print(f"R²: {r2:.3f} (yoo close)")

# Save model
joblib.dump(model, 'kigali_house_model.pkl')
print("Model saved")

ModuleNotFoundError: No module named 'sklearn'